In [63]:
# Import the pandas library for data manipulation
import pandas as pd
import xgboost as xgb
import numpy as np
import warnings
from sklearn2pmml import PMMLPipeline, sklearn2pmml
warnings.filterwarnings("ignore")
 
# Load the data from a CSV file into a DataFrame
tdf = pd.read_csv('CCF_data/card_transaction.v1.csv',index_col=False)

In [65]:
# Separating features (X_tdf) and target variable (y_tdf)
X_tdf = tdf.drop("Is Fraud?", axis=1)
y_tdf = tdf["Is Fraud?"]

In [66]:
# Converting selected columns to string type
X_tdf['Errors?'] = X_tdf['Errors?'].astype(str)
X_tdf['Merchant State'] = X_tdf['Merchant State'].astype(str)
X_tdf['Merchant Name'] = X_tdf['Merchant Name'].astype(str)
# X_tdf['Zip'] = X_tdf['Zip'].astype(pd.Int64Dtype()).astype(str)
X_tdf['Zip'] = X_tdf['Zip'].astype(str)

In [73]:
# Encoding the target variable using LabelEncoder
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_tdf_encoded = label_encoder.fit_transform(y_tdf)

# Displaying the encoded target variable
y_tdf_encoded

array([0, 0, 0, ..., 0, 0, 0])

In [74]:
def time_to_minutes(time):
    k=time.split(':')
    return int(k[0])*60+int(k[1])

In [75]:
X_tdf['Time']=X_tdf["Time"].apply(lambda x:time_to_minutes(x))

In [76]:
def remove_dollar(amount):
    return float(amount.replace('$',''))

In [77]:
X_tdf['Amount']=X_tdf["Amount"].apply(lambda x:remove_dollar(x))

In [78]:
xgb_model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='auc',
    use_label_encoder=False,
    random_state=42
) 

In [79]:
# Defining a pipeline consisting of data preprocessing
from sklearn_pandas import DataFrameMapper
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

xgb_pipeline = PMMLPipeline([
    ('mapper', DataFrameMapper([
        # Scaling numerical features
        (['User', 'Card', 'Year', 'Month', 'Day', 'Time', 'Amount'], StandardScaler()),
        # Encoding categorical features
        (['Use Chip'], LabelEncoder()),
        (['Merchant Name'], LabelEncoder()),
        (['Merchant City'], LabelEncoder()),
        (['Merchant State'], [SimpleImputer(strategy='constant', fill_value='missing'), LabelEncoder()]),
        (['Zip'], [SimpleImputer(strategy='constant', fill_value='missing'), LabelEncoder()]),
        (['MCC'], LabelEncoder()),
        (['Errors?'], [SimpleImputer(strategy='constant', fill_value='missing'), LabelEncoder()]),
      ])),
    ("classifier", xgb_model)
])

In [81]:
X_tdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24386900 entries, 0 to 24386899
Data columns (total 14 columns):
 #   Column          Dtype  
---  ------          -----  
 0   User            int64  
 1   Card            int64  
 2   Year            int64  
 3   Month           int64  
 4   Day             int64  
 5   Time            int64  
 6   Amount          float64
 7   Use Chip        object 
 8   Merchant Name   object 
 9   Merchant City   object 
 10  Merchant State  object 
 11  Zip             object 
 12  MCC             int64  
 13  Errors?         object 
dtypes: float64(1), int64(7), object(6)
memory usage: 2.5+ GB


In [82]:
xgb_pipeline.fit(X_tdf,y_tdf_encoded)

PMMLPipeline(steps=[('mapper', DataFrameMapper(drop_cols=[],
                features=[(['User', 'Card', 'Year', 'Month', 'Day', 'Time',
                            'Amount'],
                           StandardScaler()),
                          (['Use Chip'], LabelEncoder()),
                          (['Merchant Name'], LabelEncoder()),
                          (['Merchant City'], LabelEncoder()),
                          (['Merchant State'],
                           [SimpleImputer(fill_value='missing',
                                          strategy='constant'),
                            LabelEncoder()]),
                          (['Zip'],
                           [SimpleImputer(fill_value='missing',
                                          strategy='constant'),
                            LabelEncoder()]),
                          (['MCC'], LabelEncoder()),
                          (['Errors?'],
                           [SimpleImputer(fill_value='missing',
                                          strategy='constant'),
                            LabelEncoder()])])),
       ('classifier', XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=7, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1000, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...))])

In [83]:
X_tdf

,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?
0,0,0,2002,9,1,381,134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,nan
1,0,0,2002,9,1,402,38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,nan
2,0,0,2002,9,2,382,120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,nan
3,0,0,2002,9,2,1065,128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,nan
4,0,0,2002,9,3,383,104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24386895,1999,1,2020,2,27,1343,-54.00,Chip Transaction,-5162038175624867091,Merrimack,NH,3054.0,5541,nan
24386896,1999,1,2020,2,27,1344,54.00,Chip Transaction,-5162038175624867091,Merrimack,NH,3054.0,5541,nan
24386897,1999,1,2020,2,28,463,59.15,Chip Transaction,2500998799892805156,Merrimack,NH,3054.0,4121,nan
24386898,1999,1,2020,2,28,1210,43.12,Chip Transaction,2500998799892805156,Merrimack,NH,3054.0,4121,nan


In [84]:
sklearn2pmml(xgb_pipeline, "xgboost_fraud_detection_2.pmml", with_repr=True)

In [85]:
from pypmml import Model

In [86]:
pmml_model = Model.load('xgboost_fraud_detection.pmml')

In [87]:
X_tdf.head()

,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?
0,0,0,2002,9,1,381,134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,nan
1,0,0,2002,9,1,402,38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,nan
2,0,0,2002,9,2,382,120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,nan
3,0,0,2002,9,2,1065,128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,nan
4,0,0,2002,9,3,383,104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,nan


In [91]:
X_tdf["Errors?"].unique()

array(['nan', 'Technical Glitch,', 'Insufficient Balance,', 'Bad PIN,',
       'Bad PIN,Insufficient Balance,', 'Bad Expiration,',
       'Bad PIN,Technical Glitch,', 'Bad Card Number,', 'Bad CVV,',
       'Bad Zipcode,', 'Insufficient Balance,Technical Glitch,',
       'Bad Card Number,Insufficient Balance,',
       'Bad Card Number,Bad CVV,', 'Bad CVV,Insufficient Balance,',
       'Bad Card Number,Bad Expiration,', 'Bad Expiration,Bad CVV,',
       'Bad Expiration,Insufficient Balance,',
       'Bad Expiration,Technical Glitch,',
       'Bad Card Number,Bad Expiration,Technical Glitch,',
       'Bad CVV,Technical Glitch,', 'Bad Card Number,Technical Glitch,',
       'Bad Zipcode,Insufficient Balance,',
       'Bad Zipcode,Technical Glitch,',
       'Bad Card Number,Bad Expiration,Insufficient Balance,'],
      dtype=object)

In [89]:
X_tdf["Merchant State"].unique()

array(['CA', 'nan', 'NE', 'IL', 'MO', 'Switzerland', 'IA', 'TX',
       'Estonia', 'NJ', 'NV', 'NY', 'Japan', 'AZ', 'UT', 'FL', 'MI',
       'Mexico', 'WA', 'OH', 'Dominican Republic', 'NM', 'China', 'SC',
       'AK', 'PA', 'VA', 'Portugal', 'HI', 'CT', 'MA', 'MN', 'CO',
       'Italy', 'GA', 'Philippines', 'Jamaica', 'AR', 'Canada', 'OR',
       'WI', 'NC', 'France', 'WV', 'Norway', 'ME', 'NH', 'VT', 'MD', 'AL',
       'KY', 'TN', 'MS', 'New Zealand', 'Netherlands', 'KS', 'ND', 'DC',
       'Sierra Leone', 'Haiti', 'MT', 'South Korea', 'Spain', 'OK',
       'Algeria', 'Egypt', 'WY', 'ID', 'Indonesia', 'RI',
       'United Kingdom', 'Colombia', 'South Africa', 'IN', 'The Bahamas',
       'Singapore', 'Pakistan', 'Malaysia', 'Mongolia', 'Israel',
       'Hong Kong', 'Guatemala', 'Peru', 'Thailand', 'Sweden', 'Taiwan',
       'LA', 'Germany', 'Czech Republic', 'India', 'Poland', 'DE',
       'Costa Rica', 'Denmark', 'Bangladesh', 'Macedonia', 'Nigeria',
       'SD', 'Tuvalu', 'Ireland',

In [90]:
X_tdf["Zip"].unique()

array(['91750.0', '91754.0', '91755.0', ..., '5440.0', '12028.0',
       '12958.0'], dtype=object)